In [1]:
import numpy as np
import os, sys 
import re
import ROOT
ROOT.gStyle.SetOptStat(0)
import glob
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

CDMS = os.environ["CDMS"] # set in .bash_profile

sys.path.append(os.path.join(os.path.join(CDMS,"scripts")))
from ROOplot import ROOplot

Welcome to JupyROOT 6.28/10


In [2]:
# assume that lambda_a = 0 in these clean room environments
lambda_0 = 0.00755    # Rn222 decay constant [/h]
lambda_1 = 13.37      # Po218 decay constant [/h]
lambda_2 = 1.552      # Pb214 decay constant [/h]
lambda_3 = 2.111      # Bi214 decay constant [/h]

In [3]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

dt = datetime.now(ZoneInfo("America/New_York"))

In [4]:
# Heavy-etched ended on 2018-07-19 17:42 where it was then heat sealed in a nylon bag at SUF polishing storage.
locations_txt = [("SUF Polishing (Tunnel A)", "2018-07-19 12:14:00.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-07-19 23:15:24.0"),
             ("SUF Polishing (Tunnel A)", "2018-07-20 18:21:47.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-07-21 00:48:27.0"),
             ("SUF Polishing (Tunnel A)", "2018-07-23 15:53:55.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-07-23 22:14:55.0"),
             ("SUF Polishing (Tunnel A)", "2018-07-24 18:43:36.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-07-25 00:24:47.0"),
             ("SUF Polishing (Tunnel A)", "2018-07-25 16:49:12.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-07-25 23:03:15.0"),
             ("SUF Polishing (Tunnel A)", "2018-07-26 17:02:08.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-07-26 23:37:35.0"),
             ("SUF Polishing (Tunnel A)", "2018-07-31 16:05:25.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-01 01:50:04.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-01 16:24:35.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-01 23:16:07.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-02 16:08:58.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-02 23:28:32.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-03 17:24:34.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-03 21:23:23.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-03 21:54:33.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-03 22:06:36.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-06 17:05:49.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-06 23:11:13.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-07 17:42:11.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-08 00:36:47.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-08 18:01:00.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-08 23:29:27.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-09 16:48:42.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-09 23:30:43.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-10 16:49:20.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-10 19:50:34.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-13 16:12:34.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-13 23:06:10.0"),
             ("SUF Polishing (Tunnel A)", "2018-08-15 22:03:36.0"),
             ("SUF Polishing Storage (Tunnel A)", "2018-08-15 22:16:24.0"),
             ("SUF Polishing (Tunnel A)", "2018-09-21 23:10:19.0"),
             ("SUF Storage (Tunnel C)", "2018-09-21 23:15:51.0"),
             ("SNF Lab", "2019-11-07 18:21:26.0"),
             ("B04 Lab", "2019-11-08 01:11:43.0"),
             ("SUF Storage (Tunnel C)", "2019-11-09 04:47:10.0"),
             ("SNF Lab", "2019-11-14 23:09:22.0"),
             ("SUF Storage (Tunnel C)", "2019-11-15 00:54:49.0"),
             ("SNF Lab", "2019-11-15 23:02:26.0"),
             ("SUF Storage (Tunnel C)", "2019-11-16 04:10:53.0"),
             ("SNF Lab", "2019-11-19 01:17:22.0"),
             ("SUF Storage (Tunnel C)", "2019-11-19 02:17:58.0"),
             ("SNF Lab", "2019-11-21 22:14:36.0"),
             ("SUF Storage (Tunnel C)", "2019-11-23 04:48:11.0"),
             ("SNF Lab", "2019-11-25 20:26:10.0"),
             ("SUF Storage (Tunnel C)", "2019-11-25 22:26:59.0"),
             ("SNF Lab", "2019-11-27 21:18:51.0"),
             ("SUF Storage (Tunnel C)", "2019-11-28 02:28:19.0"),
             ("SNF Lab", "2019-12-02 23:50:04.0"),
             ("SUF Storage (Tunnel C)", "2019-12-03 01:24:23.0"),
             ("SNF Lab", "2019-12-06 18:00:57.0"),
             ("SUF Storage (Tunnel C)", "2019-12-07 02:29:29.0"),
             ("SNF Lab", "2019-12-09 21:07:43.0"),
             ("SUF Storage (Tunnel C)", "2019-12-09 23:49:48.0"),
             ("SNF Lab", "2019-12-10 19:03:33.0"),
             ("SUF Storage (Tunnel C)", "2019-12-10 21:39:30.0"),
             ("SNF Lab", "2019-12-11 22:14:40.0"), # Dry etching, assume no benefit
             ("SUF Storage (Tunnel C)", "2019-12-12 02:24:32.0"),
             ("SNF Lab", "2019-12-13 23:40:19.0"),
             ("SUF Storage (Tunnel C)", "2019-12-14 00:53:19.0"),
             ("SNF Lab", "2022-05-11 18:34:56.0"),
             ("SUF Purged Storage (Tunnel C)", "2022-05-12 03:28:22.0"),
             ("RSF Lab", "2022-06-29 21:56:14.0"),
             ("SUF Purged Storage (Tunnel C)", "2022-06-30 04:25:12.0"),
             ("RSF Lab", "2022-07-20 21:09:20.0"),
             ("SUF Purged Storage (Tunnel C)", "2022-07-21 02:21:41.0"),
             ("SUF Purged Storage (Tunnel C)", "2022-07-21 02:26:45.0"),
             ("RSF Lab", "2022-07-22 06:00:05.0"),
             ("SUF Purged Storage (Tunnel C)", "2022-07-22 06:00:15.0"),
             ("B33 Laminar flow bench", "2022-07-28 15:04:37.0"),
             ("B33 Purged Storage", "2022-07-28 15:58:42.0"),
             ("B33 Clean Room", "2022-07-28 20:51:57.0"),
             ("B33 Bluefors", "2022-07-28 23:39:38.0"),
             ("B33 Clean Room", "2022-07-29 16:10:11.0"),
             ("B33 Vacuum", "2022-07-29 21:56:43.0"),
             ("B33 Clean Room", "2022-08-08 13:15:15.0"),
             ("B33 Laminar flow bench", "2022-08-08 18:01:26.0"),
             ("B33 Tower Storage Container", "2022-08-08 20:51:02.0"),
             ("SLC South Adit", "2022-08-10 04:26:24.0"),
             ("SLAC_to_SNOLAB", "2023-05-09 16:23:32.0"),
             ("SNOLAB Surface", "2023-05-12 18:16:22.0"),
             ("Underground", "2023-05-13 13:08:48.0"),
             ("Underground Storage Container", "2023-05-14 22:09:26.0"),
             ("Low Radon Cleanroom", "2023-05-23 21:30:58.0"),
             ("Underground Storage Container", "2023-05-23 22:32:22.0"),
             ("Low Radon Cleanroom", "2023-05-30 16:22:22.0"),
             ("Underground Storage Container", "2023-05-31 13:12:27.0"),
             #("Low Radon Cleanroom", "2023-10-11 13:45:31.0"), # I don't understand this timestamp. CUTE infrastructure elog/66
             ("Low Radon Cleanroom", "2023-10-11 10:29:00.0"), # removed from container, CUTE infrastructure elog/66
             ("LRCR purge bag", "2023-10-11 12:35:00.0"), # Purging during lunch break, CUTE infrastructure elog/66
             ("Low Radon Cleanroom", "2023-10-11 13:30:00.0"), # removed from purge bag, CUTE infrastructure elog/66
             ("LRCR purge bag", "2023-10-11 13:34:00.0"), # Purging, CUTE infrastructure elog/66
             ("Low Radon Cleanroom", "2023-10-12 08:44:00.0"), # removed from purge bag, elog/177
             ("LRCR purge bag", "2023-10-12 12:07:00.0"), # break for lunch, later started moving to CUTE in this config, elog/177
             #("CUTE UG", "2023-10-12 18:17:24.0"), # not quite accurate, excluding this timestamp.
             ("CUTE UG", "2023-10-14 08:38:00.0"), # removed from purge bag, CUTE operations elog/179
             ("CUTE UG purge bag", "2023-10-14 12:50:00.0"), # break for lunch, CUTE operations elog/179
             ("CUTE UG", "2023-10-14 13:55:00.0"), # removed from purge bag, CUTE operations elog/179
             ("CUTE UG purge bag", "2023-10-14 15:00:00.0"), # assume purge bag was put back on at 3pm, CUTE operations elog/179
             ("CUTE UG", "2023-10-16 08:25:00.0"), # removed from purge bag, CUTE operations elog/180
             ("CUTE UG purge bag", "2023-10-16 12:51:00.0"), # lunch break, CUTE operations elog/180
             ("CUTE UG", "2023-10-16 14:10:00.0"), # removed from purge bag, CUTE operations elog/180
             ("CUTE UG purge bag", "2023-10-16 15:07:00.0"), # bagging up at end of shift, CUTE operations elog/180
             ("CUTE UG", "2023-10-18 08:22:00.0"), # removed from purge bag, CUTE operations elog/181
             ("CUTE UG purge bag", "2023-10-18 12:10:00.0"), # lunch break, CUTE operations elog/181
             ("CUTE UG", "2023-10-18 13:23:00.0"), # removed from purge bag, CUTE operations elog/181
             ("CUTE UG purge bag", "2023-10-18 15:06:00.0"), # bagging up at end of shift, CUTE operations elog/181
             ("CUTE UG", "2023-10-19 08:24:00.0"), # removed from purge bag, CUTE operations elog/182
             ("SNOLAB CUTE Vacuum", "2023-10-19 15:23:00.0"), # <1 mbar in CUTE fridge, CUTE operations elog/182. Edit to original timestamp which was the 20th.
             ("CUTE UG purge bag", "2024-03-18 14:56:00.0"), # bagging up after removing from drywell, CUTE detectors elog/76
             ("CUTE UG", "2024-03-19 08:19:00.0"), # removed from purge bag, CUTE detectors elog/77
             ("CUTE UG purge bag", "2024-03-19 12:17:00.0"), # lunch break, CUTE detectors elog/77
             ("CUTE UG", "2024-03-19 13:35:00.0"), # removed from purge bag, CUTE detectors elog/77
             ("CUTE UG purge bag", "2024-03-19 14:26:00.0"), # bagging up, CUTE detectors elog/77
             ("CUTE UG", "2024-03-20 09:06:00.0"), # removed from purge bag, CUTE detectors elog/78
             ("CUTE UG purge bag", "2024-03-20 10:24:00.0"), # bagging up, CUTE detectors elog/78
             ("LRCR purge bag", "2024-03-20 11:02:00.0"), # purge bag moved into cleanroom, CUTE detectors elog/78
             ("Low Radon Cleanroom", "2024-03-20 13:52:00.0"), # changed time based on CUTE detectors elog/78
             ("Underground Storage Container", "2024-03-20 15:35:00.0"), # changed date and time based on CUTE detectors elog/78
             ("Low Radon Cleanroom", "2025-03-18 12:21:27.0"),
             ("LRCR purge bag", "2025-03-18 18:18:29.0"),
             ("Low Radon Cleanroom", "2025-03-19 16:22:38.0"),
             ("LRCR purge bag", "2025-03-19 17:42:10.0"),
             ("Low Radon Cleanroom", "2025-03-20 12:44:01.0"),
             ("LRCR purge bag", "2025-03-20 16:31:32.0"),
             ("Low Radon Cleanroom", "2025-04-01 15:59:29.0"),
             ("Purged SNOBOX", "2025-07-17 09:00:00.0"), # added
             ("Vacuum SNOBOX", "2025-10-14 11:40:00.0"), # added
             ("Vacuum SNOBOX", "2026-03-03 00:00:00.0")] # added

In [5]:
locations_unix=[(i[0], int(datetime.strptime(i[1], "%Y-%m-%d %H:%M:%S.%f").timestamp())) for i in locations_txt]
locations_elapsed = [(locations_unix[i][0], (locations_unix[i+1][1] - locations_unix[i][1])) for i in range(len(locations_unix)-1)]

In [25]:
####      |           Location                 | Activity (Bq/m3) | ventilation rate (/h) | surface area fraction |      S V^{-1}    |
mapping = {"SUF Polishing (Tunnel A)":         {"A_0": 25,        "lambda_v": 195,        "det_frac": 0.0003,       "SV": 3},
           "SUF Polishing Storage (Tunnel A)": {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "SUF Storage (Tunnel C)":           {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "SNF Lab":                          {"A_0": 12,        "lambda_v": 195,        "det_frac": 0.0003,       "SV": 3},
           "B04 Lab":                          {"A_0": 20,        "lambda_v": 195,        "det_frac": 0.0003,       "SV": 3},
           "SUF Purged Storage (Tunnel C)":    {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "RSF Lab":                          {"A_0": 1.883,     "lambda_v": 360,        "det_frac": 0.0003,       "SV": 3},
           "B33 Laminar flow bench":           {"A_0": 15,        "lambda_v": 420,        "det_frac": 0.0003,       "SV": 3},
           "B33 Purged Storage":               {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "B33 Clean Room":                   {"A_0": 15,        "lambda_v": 195,        "det_frac": 0.0003,       "SV": 3},
           "B33 Bluefors":                     {"A_0": 0,         "lambda_v": 0,          "det_frac": 0.1,          "SV": 120},
           "B33 Vacuum":                       {"A_0": 0,         "lambda_v": 0,          "det_frac": 0.1,          "SV": 120},
           "B33 Tower Storage Container":      {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "SLC South Adit":                   {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "SLAC_to_SNOLAB":                   {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "SNOLAB Surface":                   {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "Underground":                      {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "Underground Storage Container":    {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "Low Radon Cleanroom":              {"A_0": 1,         "lambda_v": 34,         "det_frac": 0.0003,       "SV": 2},
           "LRCR purge bag":                   {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "CUTE UG":                          {"A_0": 10,        "lambda_v": 0.06,       "det_frac": 0.0003,       "SV": 3},
           "CUTE UG purge bag":                {"A_0": 0.01,      "lambda_v": 420,        "det_frac": 0.1,          "SV": 120},
           "SNOLAB CUTE Vacuum":               {"A_0": 0,         "lambda_v": 0,          "det_frac": 0.1,          "SV": 120},
           "Purged SNOBOX":                    {"A_0": 0.01,      "lambda_v": 0.1,        "det_frac": 0.0003,       "SV": 3},
           "Vacuum SNOBOX":                    {"A_0": 0,         "lambda_v": 0,          "det_frac": 0.0003,       "SV": 3}}

The balance equation for Po218 concentration in the air is $\lambda_1 A_0 = (\lambda_1 + \lambda_v + \lambda_d) A_1$

The balance equation for Po218 concentration on a surface is $\lambda_d A_1 = \lambda_1 A_1^{S*}$

The balance equation for Pb214 concentration in the air is $\lambda_2 A_1 = (\lambda_2 + \lambda_v + \lambda_d) A_2$

The balance equation for Pb214 concentration on a surface is $\lambda_2 A_1^{S*} + \lambda_d A_2 = \lambda_2 A_2^{S*}$

In [26]:
def calc_implantation_rate(A_0, lambda_v, det_frac, SV, frac_implanted=0.6):

    v_d = 8 # m/h
    lambda_d = SV * v_d # /h
    lambda_d_det = lambda_d * det_frac # /h
    
    #### Steady-state of Po-218 in air
    # source: decay of Rn-222 in air
    # sinks: decay of Po-218 in air, ventilation, deposition
    A_1 = lambda_1 * A_0 / (lambda_1 + lambda_v + lambda_d)

    #### Steady-state of Po-218 on surfaces
    # source: deposition of airborne Po-218
    # sinks: decay of deposited Po-218 on surfaces
    A_1_s = lambda_d * A_1 / lambda_1
    #### Steady-state of Po-218 on detector surfaces
    # source: deposition of airborne Po-218 onto a detector surface
    # sinks: decay of Po-218 on detector surfaces
    A_1_s_det = lambda_d_det * A_1 / lambda_1

    #### Steady-state of Pb-214 in air
    # source: decay of Po-218 in air
    # sinks: decay of Pb-214 in air, ventilation, deposition
    A_2 = lambda_2 * A_1 / (lambda_2 + lambda_v + lambda_d)

    #### Steady-state of Pb-214 on surfaces
    # source: decay of Po-218 on surfaces, deposition of airborne Pb-214
    # sinks: decay of Pb-214 on surfaces
    A_2_s = (lambda_2 * A_1_s + lambda_d * A_2) / lambda_2
    #### Steady-state of Pb-214 on detector surfaces
    # source: decay of Po-218 that was on a detector surface, deposition of airborne Pb-214 onto a detector surface
    # sinks: decay of Pb-214 on detector surfaces
    A_2_s_det = (lambda_2 * A_1_s_det + lambda_d_det * A_2) / lambda_2

    #### Steady-state of Bi-214 in air
    # source: decay of Pb-214 in air
    # sinks: decay of Bi-214 in air, ventilation, deposition
    A_3 = lambda_3 * A_2 / (lambda_3 + lambda_v + lambda_d)
    
    #### Steady-state of Bi-214 on surfaces
    # source: decay of Pb-214 on surfaces, deposition of airborne Bi-214
    # sinks: decay of Bi-214 on surfaces
    A_3_s = (lambda_3 * A_2_s + lambda_d * A_3) / lambda_3
    #### Steady-state of Bi-214 on detector surfaces
    # source: decay of Pb-214 that was on a detector surface, deposition of airborne Bi-214 onto a detector surface
    # sinks: decay of Bi-214 on detector surfaces
    A_3_s_det = (lambda_3 * A_2_s_det + lambda_d_det * A_3) / lambda_3

    #### Of the Pb-214 nuclei on surfaces, only a fraction implant into a surface
    #### due to the recoil of its decay or the previous decay
    R_implanted = A_3_s_det * frac_implanted / SV # convert from volume-equivalent activity to surface activity
    
    return R_implanted

In [27]:
#Pb210 radioactive properties
Pb210_halflife = 7.001e8 # s
Pb210_lifetime = Pb210_halflife / np.log(2) # s
Pb210_lambda   = 1 / Pb210_lifetime

$N_{\text{Pb-210}} = N_0 \text{exp}(-\lambda_{\text{Pb-210}} T) + \int_0^T R_{\text{implanted}} \cdot \text{exp}(-\lambda_{\text{Pb-210}} (T - t)) dt$

$N_{\text{Pb-210}} = N_0 \text{exp}(-\lambda_{\text{Pb-210}} T) + R \frac{1 - \text{exp}(-\lambda_{\text{Pb-210}} T)}{\lambda_{\text{Pb-210}}}$

In [28]:
# find total implanted Pb-210 taking into account its decay
# t from t0 to T
def integrate_decays(R, N0, T):
    return R*(1 - np.exp(-Pb210_lambda * T))/Pb210_lambda + N0 * np.exp(-Pb210_lambda * T)

In [29]:
# initial number of implanted Pb-210
N0 = 0
s2day = 1.157e-5
s2month = 3.80517e-7

# Window for plotting
window_start = locations_unix[0][1]
window_end = datetime(2026, 3, 3, 0, 0, 0).timestamp()
# data-taking window
datataking_start = datetime(2023, 12, 16, 0, 0, 0, tzinfo=ZoneInfo("America/New_York")).timestamp()
datataking_end = datetime(2024, 1, 9, 0, 0, 0, tzinfo=ZoneInfo("America/New_York")).timestamp()

plt.figure(figsize=(5,3), dpi=2000)

for i, location in enumerate(locations_txt[:-1]):
    A_0, lambda_v, det_frac, SV = mapping[location[0]]["A_0"], mapping[location[0]]["lambda_v"], mapping[location[0]]["det_frac"], mapping[location[0]]["SV"]
    R_implanted = calc_implantation_rate(A_0, lambda_v, det_frac, SV, frac_implanted=0.558) # Pb-214 decays / s / m^2
    #print(f"{location[0]} {R_implanted}")

    # number of implanted nuclei (N) vs. time (t)
    t0 = locations_unix[i][1]
    tf = locations_unix[i][1] + locations_elapsed[i][1]
    t = np.linspace(t0, tf, 100)
    N = integrate_decays(R_implanted, N0, t - t0)

    #### plot activations vs. time at start of window ####
    plt.plot((t - window_start)*s2month, N*1e-4, color = 'black', lw = 1)
    N0 = N[-1]

    if (i == 0):
        plt.text((t0 - window_start)*s2month, 0.02e-1, "SUF Polishing", rotation=80, fontsize = 4, ha='right', va='bottom', color = 'blue')
    if (i == 35):
        plt.text((tf - window_start)*s2month+2, 0.1e-1, "SUF Storage", rotation=20, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 63):
        plt.text((tf - window_start)*s2month-3, 0.2e-1, "SNF Lab", rotation=68, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 64):
        plt.text((tf - window_start)*s2month-17, 0.33e-1, "SUF Storage", rotation=20, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 77):
        plt.text((tf - window_start)*s2month-4, 0.45e-1, "B33", rotation=45, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 83):
        plt.text((t0 - window_start)*s2month+1, 0.5e-1, "SLC South Adit"+"\n"+"SLAC to SNOLAB", rotation=21, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 93):
        plt.text((t0 - window_start)*s2month-1.5, 0.58e-1, "CUTE CR", rotation=90, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 110):
        print(f"Activity on detector surface at start of CUTE run is {N[-1]*Pb210_lambda*1e-4*1e9:.2g} nBq/cm^2")
    if (i == 111):
        plt.text((t0 - window_start)*s2month, 0.63e-1, "CUTE" + "\n" + "Runs", rotation=-3, fontsize = 4, ha='left', va='bottom', color = 'blue')
    #if (i == 112):
        #plt.text((t0 - window_start)*s2month-1, 1.37, "CUTE CR", rotation=90, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 121):
        plt.text((t0 - window_start)*s2month+6, 0.65e-1, "Underground" + "\n" + "Storage", rotation=23, fontsize = 4, ha='center', va='bottom', color = 'blue')
    if (i == 128):
        plt.text((t0 - window_start)*s2month-1, 0.75e-1, "Installation", rotation=70, fontsize = 4, ha='left', va='bottom', color = 'blue')
    if (i == 129):
        print(f"Activity on detector surface when in SNOBOX vacuum is {N[-1]*Pb210_lambda*1e-4*1e9:.2g} nBq/cm^2")
    if (i == 130):
        plt.text((t0 - window_start)*s2month+1, 0.95e-1, "SNOBOX" + "\n" + "Purge" + "\n" + "+ Vacuum", rotation=-5, fontsize = 4, ha='center', va='bottom', color = 'blue')

plt.xlim(-5, (window_end-window_start)*s2month)
plt.ylim(0, 0.13)
plt.grid(True, color = 'black', lw = 0.2, ls = 'dotted')
plt.xlabel(f'Time since {datetime.fromtimestamp(window_start).strftime("%Y/%m/%d")} [months]', loc = 'right')
plt.ylabel(r'N$_{\text{Pb-210}}$ / cm$^2$', loc = 'top')
plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
plt.tight_layout()

plt.savefig("G157_hist.png")

Activity on detector surface at start of CUTE run is 0.062 nBq/cm^2
Activity on detector surface when in SNOBOX vacuum is 0.094 nBq/cm^2


In [11]:
# initial number of implanted Pb-210
N0 = 0
A_0, lambda_v, det_frac, SV = mapping["SUF Polishing (Tunnel A)"]["A_0"], mapping["SUF Polishing (Tunnel A)"]["lambda_v"], mapping["SUF Polishing (Tunnel A)"]["det_frac"], mapping["SUF Polishing (Tunnel A)"]["SV"]
R_implanted = calc_implantation_rate(A_0, lambda_v, det_frac, SV, frac_implanted=0.558) # Pb-214 decays / s / m^2

t0 = 0
tf = 60*60
t = np.linspace(t0, tf, 100)
N = integrate_decays(R_implanted, N0, t - t0)

In [12]:
print(f"Activity on detector surface is {N[-1]*Pb210_lambda*1e-4*1e9:.2g} nBq/cm^2")

Activity on detector surface is 5.4e-05 nBq/cm^2
